# oyLabImaging Spatial Statistics Pipeline
## Multi-Timepoint (Timelapse) Example

This script demonstrates how to extract, calculate, and visualize spatiotemporal interactions from segmented microscopy data. It covers:
1. Global Spatial Autocorrelation (Moran's I, Geary's C)
2. Local Spatial Hotspot Detection (Local Moran's I)
3. Spatial Expression Mapping
4. Categorical Neighborhood Enrichment
5. Interactive Napari Visualization (Snapshots & Timelapse Movies)

In [ ]:
%load_ext autoreload
%autoreload 2
%gui qt
%matplotlib inline

import sys
import os
import dill
import numpy as np
import pandas as pd
import anndata as ad

# Point to our modified package directory
sys.path.insert(0, "../oyLabImaging")
from oyLabImaging import Metadata
from oyLabImaging.Processing.Results import results

### 1. Data Loading & Environment Setup

In [ ]:
# Define the path to the dataset
data_path = '/bigstore/Microscopy Core/Jen/3T3_mRubyloss_HSV_20210818/'
print("Loading results.pickle (This contains all timepoints!)...")

# Load the master results object
with open(os.path.join(data_path, "results.pickle"), "rb") as f:
    R = dill.load(f)
R.pth = data_path

# Load the individual Position files (PosLbls) to get the single-cell data
for pos_name in R.PosNames:
    pkl_file = os.path.join(data_path, "PosLbls", f"{pos_name}.pkl")
    if os.path.exists(pkl_file):
        with open(pkl_file, "rb") as f:
            P = dill.load(f)
        P.pth = data_path
        R.PosLbls[pos_name] = P

# For this test, we will only analyze one specific position
test_positions = ['B5-Site_0']

print(f"✓ Loaded successfully")
print(f"  Positions we are testing: {test_positions}")
print(f"  Timepoints found: {len(R.frames)}")
print(f"  Exact Channels found: {list(R.channels)}")

### 2. Discrete Cell Classification (Quadrant Gating)
Neighborhood Enrichment is designed for discrete cell types. We mimic flow-cytometry "Quadrant Gating" by finding the top 10% of expressors for each marker and categorizing every cell.

In [ ]:
for pos in test_positions:
    # 1. Find the global 90th percentile for the markers across the whole movie
    all_green = np.concatenate([R.PosLbls[pos].mean('Green')[t] for t in range(len(R.frames)) if R.PosLbls[pos].num[t] > 0])
    all_red = np.concatenate([R.PosLbls[pos].mean('Red')[t] for t in range(len(R.frames)) if R.PosLbls[pos].num[t] > 0])
    
    green_90th = np.percentile(all_green, 90)
    red_90th = np.percentile(all_red, 90)

    # 2. Classify each cell based on these biological thresholds
    for t in range(len(R.frames)):
        if R.PosLbls[pos].num[t] > 0:
            g_vals = R.PosLbls[pos].mean('Green')[t]
            r_vals = R.PosLbls[pos].mean('Red')[t]
            
            states = []
            for g, r in zip(g_vals, r_vals):
                if g > green_90th and r > red_90th:
                    states.append('Double+')    # High Virus, High Host
                elif g > green_90th:
                    states.append('Virus+')     # High Virus only
                elif r > red_90th:
                    states.append('Host+')      # High Host only
                else:
                    states.append('Low')        # Background/Bystander cells
                    
            R.PosLbls[pos].framelabels[t].regionprops['Cell_State'] = states

### 3. Run Spatial Statistics Over Time
Calculates metrics across all frames in the timelapse.

In [ ]:
target_channels = ['Green', 'Red']
biv_pairs = [('Green', 'Red')]

print("\n--- Running Spatial Stats Over Time ---")
R.calculate_spatial_stats(
    Position=test_positions, 
    metrics=[
        'morans_i', 
        'gearys_c', 
        'neighborhood_enrichment', 
        'bivariate_moran', 
        'local_morans_i', 
        'local_bivariate_moran'
    ],
    channels=target_channels,
    bivariate_pairs=biv_pairs,
    cluster_key='Cell_State',          # Uses the discrete Cell_State column we generated
    n_neighs=6,                        
    nhood_frames=[0, 20, 40, 60, 80],  # Skip frames to speed up categorical calculations
    export_h5ad=False, 
    save=False         
)

### 4. Diagnostics: View Cell Counts Over Time
Important to verify cells aren't detaching/dying off halfway through the movie.

In [ ]:
print("\n--- Diagnostic: Cell Counts per Frame ---")
counts_dict = {'Frame': R.frames}
for pos in test_positions:
    counts_dict[pos] = R.PosLbls[pos].num
    
df_counts = pd.DataFrame(counts_dict)
print(df_counts.head(10)) # Change to print(df_counts) to see all 84 frames.

### 5. Visualization: Generating Matplotlib Plots
Showcasing Global Metrics, Time-Series dynamics, and specific frame snapshots.

In [ ]:
my_colors = {
    'Green': '#2ca25f',                       # Univariate: Green (Virus)
    'Red': '#d62728',                         # Univariate: Red (Alive mRuby)
    'Green vs Red': '#9467bd',                # Bivariate: Purple (Co-localization)
    ('Virus+', 'Virus+'): '#2ca25f',          # Nhood: Virus - Virus
    ('Virus+', 'Host+'): '#9467bd',           # Nhood: Virus - Host
    ('Host+', 'Host+'): '#d62728',            # Nhood: Host - Host
}

# --- A. GLOBAL METRICS ---
print("\n--- Plotting Univariate: Moran's I ---")
R.plot_spatial_stats(Position=test_positions, metric='morans_i', channels=target_channels, custom_colors=my_colors)
print("\n--- Plotting Univariate: Geary's C ---")
R.plot_spatial_stats(Position=test_positions, metric='gearys_c', channels=target_channels, custom_colors=my_colors)
print("\n--- Plotting Bivariate Moran's I ---")
R.plot_spatial_stats(Position=test_positions, metric='bivariate_moran', custom_colors=my_colors)

# --- B. TIME-SERIES DYNAMICS ---
print("\n--- Plotting Expression Dynamics (Time Series) ---")
R.plot_spatial_stats(Position=test_positions, metric='expression', channels=target_channels, custom_colors=my_colors)
print("\n--- Plotting Local Moran's I (Time Series) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_morans_i', channels=target_channels, custom_colors=my_colors)
print("\n--- Plotting Local Bivariate Moran's I (Time Series) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_bivariate_moran', custom_colors=my_colors)

# --- C. SPATIAL SNAPSHOTS (FRAME 40) ---
print("\n--- Plotting Expression (Spatial Map Snapshot of Frame 40) ---")
R.plot_spatial_stats(Position=test_positions, metric='expression', channels=target_channels, plot_type='spatial_map', frame_idx=40)
print("\n--- Plotting Local Univariate (Spatial Map Snapshot of Frame 40) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_morans_i', channels=target_channels, plot_type='spatial_map', frame_idx=40)
print("\n--- Plotting Local Bivariate (Spatial Map Snapshot of Frame 40) ---")
R.plot_spatial_stats(Position=test_positions, metric='local_bivariate_moran', plot_type='spatial_map', frame_idx=40)

# --- D. NEIGHBORHOOD ENRICHMENT ---
print("\n--- Plotting Neighborhood Enrichment Line Graphs ---")
my_pairs = [('Virus+', 'Virus+'), ('Virus+', 'Host+'), ('Host+', 'Host+')]
R.plot_spatial_stats(Position=test_positions, metric='neighborhood_enrichment', nhood_pairs=my_pairs, custom_colors=my_colors)

### 6. Interactive Napari Visualization: Single Frame Snapshots
Overlays the spatial statistics points directly on top of the raw TIF microscopy images for a specific frame.

In [ ]:
print("\n--- Opening Napari to view Hotspots on Frame 40 ---")
viewer = R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Green', 
    metric='local_morans_i', 
    frame_idx=40,  
    size=10 
)

print("\n--- Opening Napari to view BIVARIATE Hotspots on Frame 40 ---")
viewer_biv = R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Green vs Red', 
    metric='local_bivariate_moran', 
    frame_idx=40,  
    size=10
)

### 7. Interactive Napari Visualization: Full Timelapse Movie
By passing `frame_idx='all'`, we generate a slider to view the stats change over time.
Setting `load_images=False` bypasses reading heavy TIFs, opening the movie instantly.
By stacking the commands, we load 3 separate toggleable layers into the same viewer!

In [ ]:
print("\n--- Opening Napari to view EXPRESSION & HOTSPOTS OVER TIME ---")

# Call 1: Clears the viewer, and loads the Green expression map
viewer_multi = R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Green',
    metric='expression', 
    frame_idx='all',  
    size=10,
    load_images=False,
    clear_viewer=True   
)

# Call 2: Stacks the Green Hotspots on top of the same viewer
R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Green',
    metric='local_morans_i', 
    frame_idx='all',  
    size=10,
    load_images=False 
)

# Call 3: Stacks the Green vs Red Bivariate Hotspots on top
R.show_spatial_map_napari(
    pos=test_positions[0], 
    Channel='Green vs Red',
    metric='local_bivariate_moran', 
    frame_idx='all',  
    size=10,
    load_images=False 
)